# 📘 Buổi 04: Thẩm định Hợp đồng Tự động (Contract Review Agent)
## 🚀 Jupyter Notebook Operator Panel for n8n Workflow

> **Dành cho Giảng viên & Học viên**: Notebook này đóng vai trò **Giao diện Vận hành & Giám sát Tương tác (Interactive Companion Panel)** cho Workflow n8n (`http://localhost:5678`), theo đúng thiết kế bài lab trong [`lab.md`](../lab.md).
>
> **Triết lý bài học**:
> 1. **Auto-Config & Instant Launch n8n**: Tự động bật n8n (`npx n8n start` / Docker) và nạp sẵn solution workflow `checkpoints/n8n-contract-review-solution.json` mà không mất thời gian cài đặt nút thủ công.
> 2. **Harness Engineering**: Kiểm soát AI qua cổng tất định Schema Validation (`clause.schema.json`) và Evidence Check.
> 3. **Data Privacy First**: Redaction 4 cấp che 100% PII & thông tin tài chính nhạy cảm trong n8n trước khi đẩy sang AI Cloud.
> 4. **Knowledge Base First**: Nạp Kho tri thức Red Flag (`checklist-rui-ro.md`) làm tiêu chuẩn thẩm định tất định.
---

### 🐳 Step 0: Auto-Launch & Auto-Config n8n Workflow (`npx n8n start` / Docker) (Thực hành 1)
Khởi chạy dịch vụ n8n và tự động nạp workflow giải pháp `checkpoints/n8n-contract-review-solution.json` (`http://localhost:5678`).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 🔑 GEMINI API KEY (demo K1 — dùng cho TH2 AI Extract, model gemini-3.5-flash)
#    Key: YOUR_GEMINI_API_KEY
#    Khi import workflow vào n8n, paste key này vào node
#    "TH2 - AI Extract Clauses (Gemini + KB)" (thay cho REPLACE_WITH_YOUR_GEMINI_API_KEY).
# ─────────────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path
import json

# Thêm đường dẫn thư mục test vào Python path
TEST_DIR = Path(".").resolve()
BASE_DIR = TEST_DIR.parent.resolve()
if str(TEST_DIR) not in sys.path:
    sys.path.insert(0, str(TEST_DIR))

from auto_import_n8n import auto_import_workflow
from interactive_e2e_runner import InteractiveE2ERunner

print("=" * 75)
print("⚡ BƯỚC 0: KIỂM TRA & TỰ ĐỘNG BẬT DỊCH VỤ N8N (NPX N8N START / DOCKER)")
print("=" * 75)
success = auto_import_workflow()

# Khởi tạo runner và đăng nhập vào n8n API
runner = InteractiveE2ERunner()
status = runner.check_n8n_status()
print(f"\n🌐 n8n Web UI Base URL: {status['web_ui_url']}")
print("─" * 75)
print("🔐 THÔNG TIN ĐĂNG NHẬP N8N:")
print("   📧 Email   : admin@alobase.vn")
print("   🔑 Password: Password123!")
print("─" * 75)

# Đăng nhập vào n8n REST API (POST /rest/login)
print("🔑 Đăng nhập vào n8n REST API...")
logged_in = runner.ensure_logged_in()
if logged_in:
    wf_url = runner.get_workflow_url()
    print("─" * 75)
    print(f"🚀 LINK MỞ WORKFLOW N8N ĐỂ TỰ CHẠY (CANVAS UI):\n   👉 {wf_url}")
    print("─" * 75)
    print("✅ BƯỚC 0 HOÀN TẤT: n8n đã kết nối & đăng nhập API thành công!")
else:
    print("⚠️ n8n đang chạy nhưng chưa đăng nhập được API. Các bước tiếp theo sẽ thử lại.")

### 💻 Step 1: Chạy & Trải nghiệm Ứng dụng Web App Vibe Code (ReactJS - Legal AI Guard)
Khởi chạy giao diện Web App ReactJS hiện đại kết nối trực tiếp với n8n Webhook (`http://localhost:5678/webhook/contract-review`). Hỗ trợ upload file `.docx`, `.txt`, dán hợp đồng và nhận về file Báo cáo Word (`report.docx`).

In [ ]:
import subprocess
import time
import urllib.request

APP_DIR = BASE_DIR / "app"
print("=" * 75)
print("🌐 BƯỚC 1: KHỞI CHẠY & KIỂM TRA ỨNG DỤNG VIBE CODE (REACTJS WEB APP)")
print("=" * 75)

# Kiểm tra Web App ReactJS đang chạy tại port 5173
react_url = "http://localhost:5173"
is_running = False
try:
    with urllib.request.urlopen(react_url, timeout=2) as response:
        if response.status == 200:
            is_running = True
except Exception:
    is_running = False

if is_running:
    print(f"✅ Ứng dụng ReactJS Web App (Legal AI Guard) ĐANG CHẠY tại: {react_url}")
else:
    print(f"🚀 Đang phát lệnh khởi chạy ReactJS Web App tại: {APP_DIR}...")
    proc = subprocess.Popen(
        ["npm", "run", "dev"],
        cwd=str(APP_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    time.sleep(3)
    print(f"✅ Đã khởi chạy! Vui lòng truy cập giao diện tại: {react_url}")

print("─" * 75)
print("👉 TRUY CẬP GIAO DIỆN RÀ SOÁT HỢP ĐỒNG TẠI:")
print(f"   🌐 {react_url}")
print("   🔗 Target n8n Webhook: http://localhost:5678/webhook/contract-review")
print("─" * 75)
print("💡 Hướng dẫn sử dụng trên Web App:")
print("   1. Nạp file .docx/.txt hoặc bấm 'Nạp Hợp đồng Mẫu (Red Flags Test)'")
print("   2. Nhấn 'Phân tích & Xuất Báo cáo Word' để gọi n8n Webhook")
print("   3. Trình duyệt tự động tải xuống file report.docx kết quả thẩm định!")

### 📄 Step 2: Nạp & Đọc Hợp đồng Đầu vào (Input Contract)
Nạp file Hợp đồng mẫu `templates/contract-mau-hop-dong-dich-vu.docx` chứa thông tin cá nhân (PII), mã số thuế, số tiền tài chính và 8 điều khoản dịch vụ chuẩn bị truyền vào luồng n8n.

In [ ]:
# Lấy danh sách workflow đang có trên n8n qua API
print('=' * 75)
print('📋 NẠP THÔNG TIN WORKFLOW TỪ N8N API  →  GET /api/v1/workflows')
print('=' * 75)

runner.ensure_logged_in()
workflows = runner.api.list_workflows()
print(f'  • Số workflow hiện có trên n8n: {len(workflows)}')
for wf in workflows:
    print(f"    - [{wf['id']}] {wf['name']}  (active={wf.get('active')})")

# Tìm workflow và lấy full definition
wf_id = runner.find_workflow_id()
wf_def = runner._get_workflow_api()
print(f"\n  • Workflow được chọn: '{wf_def.get('name')}' (id={wf_id})")
print(f"  • Số node trong workflow: {len(wf_def.get('nodes', []))}")

# Nội dung hợp đồng đầu vào
raw_contract = runner.contract_text
print('─' * 75)
print(f'📄 HỢP ĐỒNG ĐẦU VÀO ({len(raw_contract)} ký tự):')
print(raw_contract[:600] + '\n... [Nội dung hợp đồng tiếp theo] ...')

### 🛡️ Step 3: Vận hành Node Redaction 4 Cấp Bảo mật PII trên n8n (Thực hành 2)
Khám phá và kiểm tra Node Redaction Python Regex chạy trong n8n Workflow để che thông tin nhạy cảm trước khi đẩy dữ liệu sang AI Cloud:
- **Level 1**: Che thông tin cá nhân PII (Email `[email redact]`, SĐT `0xxx`).
- **Level 2**: Che Mã số thuế doanh nghiệp (`[MST redact]`).
- **Level 3**: Che Giá trị tài chính hợp đồng (`[giá trị redact]`).
- **Level 4 Security Gate**: Dừng workflow lập tức nếu phát hiện từ khóa tuyệt mật (`STOP_IF_SECRET`).

In [ ]:
print('=' * 75)
print('🛡️ INSPECT NODE REDACTION  →  GET /api/v1/workflows/{id}')
print('=' * 75)

# Gọi n8n API để lấy cấu hình node TH1 thật sự (v2: tên node "TH1 - Redaction 4 Cap")
runner.ensure_logged_in()
try:
    node_th1 = runner.inspect_n8n_node('TH1 - Redaction 4 Cap')
    print(f"• Node Name    : {node_th1['name']}")
    print(f"• Node Type    : {node_th1['type']}")
    print(f"• typeVersion  : {node_th1['typeVersion']}")
    node_code = node_th1['parameters'].get('jsCode', node_th1['parameters'].get('pythonCode', '(không có code)'))
    print(f'• Node Redaction code (350 ký tự đầu):')
    print(node_code[:350] + '\n...')
    print('\n✅ BƯỚC 5 HOÀN TẤT: Node Redaction đã được đọc trực tiếp từ n8n API!')
except KeyError as e:
    print(f'⚠️ Node không tìm thấy trên n8n: {e}')
    print('  → Kiểm tra tên node trong workflow hoặc chạy lại Step 0 để import.')

### 🔍 Step 4: Vận hành AI Extract & Harness Schema Validation Gate trên n8n (Thực hành 3)
Sử dụng JSON Schema `templates/clause.schema.json` làm cổng kiểm soát dữ liệu do AI bóc tách trong n8n. Nếu AI trả về thiếu trường bắt buộc hoặc sai định dạng, Code Node Python Gate trong n8n sẽ từ chối và kích hoạt vòng lặp Retry.

In [ ]:
import json

schema_file = BASE_DIR / 'templates' / 'clause.schema.json'
with open(schema_file, 'r', encoding='utf-8') as f:
    clause_schema = json.load(f)

print('=' * 75)
print('🔍 INSPECT NODE SCHEMA VALIDATION  →  GET /api/v1/workflows/{id}')
print('=' * 75)

# Gọi n8n API để lấy cấu hình node Schema Validation (v2: "TH2 - Schema Validation (clause.schema.json)")
runner.ensure_logged_in()
try:
    node_th2 = runner.inspect_n8n_node('TH2 - Schema Validation (clause.schema.json)')
    print(f"• Node Name    : {node_th2['name']}")
    print(f"• Node Type    : {node_th2['type']}")
    print(f"• Schema File  : {schema_file.name}")
    print(f"• Schema Title : {clause_schema.get('title')}")
    print(f"• Required Top-level Fields : {clause_schema.get('required')}")
    print(f"• Required Clause Fields    : {clause_schema['properties']['clauses']['items']['required']}")
    print('\n✅ BƯỚC 5 HOÀN TẤT: Node Schema Validation đã được đọc từ n8n API!')
except KeyError as e:
    print(f'⚠️ Node không tìm thấy trên n8n: {e}')

### 🧠 Step 5: Policy Review vs Kho Tri Thức Red Flag & Evidence/Omission Check trên n8n (Thực hành 4)
Đối chiếu dữ liệu điều khoản bóc tách với Kho Tri Thức Red Flag (`templates/checklist-rui-ro.md`). Workflow kiểm tra đồng thời 3 lớp:
1. **Verbatim Quote Evidence Check**: kiểm tra trích dẫn nguyên văn đối chiếu với văn bản hợp đồng gốc để phát hiện AI bịa thông tin (`hallucination_flag`).
2. **Omission Check**: bắt lỗi thiếu điều khoản bắt buộc theo chính sách doanh nghiệp (ví dụ: chấm dứt, bất khả kháng, phạt vi phạm, IP).
3. **Clause Review & Red Flag Detection**: soi các điều khoản đã có để phát hiện bẫy thật như thanh toán cảm tính, đơn phương chấm dứt/gia hạn, bồi thường không giới hạn, bảo mật vô thời hạn, chuyển giao IP trước thanh toán, phạt vượt trần 8%, cơ quan tranh chấp bất lợi.

In [ ]:
print('=' * 75)
print('🧠 INSPECT NODE EVIDENCE & OMISSION  →  GET /api/v1/workflows/{id}')
print('=' * 75)

# Gọi n8n API để lấy cấu hình node Evidence & Omission Check (v2: "TH3 - Evidence + Omission (semantics)")
runner.ensure_logged_in()
kb_file = BASE_DIR / 'templates' / 'checklist-rui-ro.md'
try:
    node_th3 = runner.inspect_n8n_node('TH3 - Evidence + Omission (semantics)')
    print(f"• Node Name    : {node_th3['name']}")
    print(f"• Node Type    : {node_th3['type']}")
    print(f"• Knowledge Base: {kb_file.name} ({kb_file.stat().st_size} bytes)")
    node_code = node_th3['parameters'].get('jsCode', node_th3['parameters'].get('pythonCode', '(không có code)'))
    print('• Evidence & Omission logic (300 ký tự đầu):')
    print(node_code[:300] + '\n...')
    print('\n✅ BƯỚC 5 HOÀN TẤT: Node Evidence & Omission đọc từ n8n API thành công!')
except KeyError as e:
    print(f'⚠️ Node không tìm thấy trên n8n: {e}')

### 📊 Step 6: Vận hành Master Pipeline & Tự động xuất Báo cáo Word trên n8n (Thực hành 5)
Kích hoạt toàn bộ luồng thẩm định hợp đồng end-to-end trên n8n, tính điểm Contract Risk Score (0-100), phân loại mức độ rủi ro, và xuất file Báo cáo Word (`report.docx`) với 3 mục bắt buộc: **điều khoản bị thiếu**, **red flags/blocking issues**, và **review điều khoản đã có kèm gợi ý chỉnh sửa**.

In [ ]:
print('=' * 75)
print('🚀 TRIGGER WORKFLOW QUA WEBHOOK  →  POST /webhook/contract-review (.docx)')
print('=' * 75)

# Đảm bảo đã login trước khi trigger
runner.ensure_logged_in()

# Kích hoạt workflow v4 (webhook + respondToWebhook): gửi contract_text, nhận report.docx
contract_docx = str(BASE_DIR / 'templates' / 'contract-mau-hop-dong-dich-vu.docx')
report_out    = str(TEST_DIR / 'report.docx')
e2e_result = runner.run_e2e_pipeline(contract_docx=contract_docx, report_out=report_out)

assert 'tong_hop' in e2e_result, '❌ Luồng E2E n8n thất bại!'
print(f"\n📊 Execution data keys: {list(e2e_result.keys())}")
exec_info = e2e_result.get('execution', {})
print(f"📡 Execution ID    : {exec_info.get('id', 'N/A')}")
print(f"📡 Execution Status: {exec_info.get('status', 'N/A')}")

# Verify report.docx mở được bằng python-docx
report_path = e2e_result.get('report_path')
if report_path:
    import docx as _docx
    doc = _docx.Document(report_path)
    n_paras = len([p for p in doc.paragraphs if p.text.strip()])
    print(f"\n📄 REPORT DOCX      : {report_path}")
    print(f"📄 Số đoạn văn bản  : {n_paras}")
    print(f"📄 5 dòng đầu báo cáo:")
    for p in [p for p in doc.paragraphs if p.text.strip()][:5]:
        print(f"     | {p.text[:80]}")
    print('\n✅ ĐÃ SINH FILE BÁO CÁO HỢP ĐỒNG DẠNG DOCX THÀNH CÔNG!')
else:
    print('\n❌ Chưa sinh được report.docx — xem log lỗi phía trên.')